In [22]:
import time
import re
import os
import logging
from datetime import datetime
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.service import Service as FirefoxService
from webdriver_manager.firefox import GeckoDriverManager

from bs4 import BeautifulSoup

# Configuration du logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [23]:
## Initialisation du navigateur

In [24]:
def get_number_of_reviews(driver):
    """
    Récupère et retourne le nombre total d'avis affiché sur la page Google Maps.

    Args:
        driver (webdriver): Instance Selenium WebDriver déjà initialisée et sur la page cible.

    Returns:
        int: Le nombre total d'avis détectés. Retourne 0 en cas d'erreur.
    """
    # XPath pour localiser l'élément contenant le nombre d'avis.
    xpath_nb_avis = "/html/body/div[2]/div[3]/div[8]/div[9]/div/div/div[1]/div[2]/div/div[1]/div/div/div[4]/div[1]/div/div[2]/div[3]"
    try:
        total_number_text = driver.find_element(By.XPATH, xpath_nb_avis).text
        # Utilisation d'une expression régulière pour extraire le nombre (ex : "1 234 avis")
        import re
        match = re.search(r"([\d\s\u202f]+)", total_number_text)
        if match:
            # Supprimer les espaces ou séparateurs (ex : "\u202f")
            number_str = match.group(1).replace(" ", "").replace("\u202f", "")
            total_number_of_reviews = int(number_str)
            logging.info(f"Nombre total d'avis détectés : {total_number_of_reviews}")
        else:
            total_number_of_reviews = 0
            logging.warning("Aucun nombre détecté dans le texte d'avis.")
    except Exception as e:
        total_number_of_reviews = 0
        logging.error(f"Erreur lors de la récupération du nombre d'avis : {e}")
    return total_number_of_reviews

In [25]:
def init_driver():
    """
    Initialise le driver Firefox via GeckoDriverManager.
    """
    service = FirefoxService(GeckoDriverManager().install())
    driver = webdriver.Firefox(service=service)
    return driver

In [26]:
## Fonctions pour simuler les clics sur les boutons

In [27]:
def click_privacy_popup(driver):
    """
    Clique sur le pop-up de confidentialité s'il est présent.
    """
    xpath = "/html/body/c-wiz/div/div/div/div[2]/div[1]/div[3]/div[1]/div[1]/form[2]/div/div/button/span[6]"
    try:
        button = driver.find_element(By.XPATH, xpath)
        button.click()
        logging.info("Pop-up de confidentialité cliqué.")
    except Exception as e:
        logging.info(f"Pop-up de confidentialité non trouvé ou déjà fermé : {e}")

def click_sort_button(driver):
    """
    Clique sur le bouton de tri.
    """
    xpath = "/html/body/div[2]/div[3]/div[8]/div[9]/div/div/div[1]/div[2]/div/div[1]/div/div/div[4]/div[8]/div[2]/button/span"
    try:
        button = driver.find_element(By.XPATH, xpath)
        button.click()
        logging.info("Bouton de tri cliqué.")
    except Exception as e:
        logging.info(f"Bouton de tri non trouvé : {e}")

def click_recent_reviews(driver):
    """
    Clique sur le bouton pour trier par avis les plus récents.
    """
    xpath = "/html/body/div[2]/div[3]/div[3]/div[1]/div[2]"
    try:
        button = driver.find_element(By.XPATH, xpath)
        button.click()
        logging.info("Bouton 'Avis récents' cliqué.")
    except Exception as e:
        logging.info(f"Bouton 'Avis récents' non trouvé : {e}")

def click_more_buttons(driver):
    """
    Clique sur tous les boutons 'Plus' pour déployer le texte complet des avis.
    """
    try:
        more_buttons = driver.find_elements(By.XPATH, '//button[normalize-space()="Plus"]')
        for idx, btn in enumerate(more_buttons, start=1):
            try:
                btn.click()
                logging.info(f"Bouton 'Plus' n°{idx} cliqué.")
                time.sleep(1)
            except Exception as e:
                logging.info(f"Impossible de cliquer sur le bouton 'Plus' n°{idx} : {e}")
    except Exception as e:
        logging.info(f"Aucun bouton 'Plus' trouvé : {e}")

In [38]:
##  Fonction pour faire défiler la zone de reviews
def scroll_reviews(driver):
    """
    Fait défiler la zone de reviews pour charger plus d'avis.
    """
    xpath = "/html/body/div[2]/div[3]/div[8]/div[9]/div/div/div[1]/div[2]/div/div[1]/div/div/div[4]"
    try:
        scrollable_div = driver.find_element(By.XPATH, xpath)
        # Effectuer plusieurs défilements pour charger les avis (ajustez le nombre de fois selon vos besoins)
        for _ in range(3):
            driver.execute_script('arguments[0].scrollTop = arguments[0].scrollHeight', scrollable_div)
            time.sleep(2)
        logging.info("Scrolling effectué.")
    except Exception as e:
        logging.info(f"Zone de scroll non trouvée : {e}")

In [29]:
def extract_reviews(driver):
    """
    Extrait les éléments d'avis de la page et renvoie une liste.
    """
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    reviews = soup.find_all("div", class_="jftiEf fontBodyMedium")
    logging.info(f"Nombre d'avis extraits : {len(reviews)}")
    return reviews

In [30]:
## test

In [31]:
url = "https://www.google.fr/maps/place/Compose+Bois-Colombes+-+Cantine+sur+mesure/@48.9071788,2.2594644,17z/data=!4m8!3m7!1s0x47e665321f7e8b8b:0xe7e1d262385f4e67!8m2!3d48.9071753!4d2.2620393!9m1!1b1!16s%2Fg%2F11tg7p7k1t?entry=ttu&g_ep=EgoyMDI0MTIxMS4wIKXMDSoASAFQAw%3D%3D"
entreprise =  "Compose"
name = "test"
nb_avis = "0"

In [32]:
driver = init_driver()
driver.get(url)    

2025-03-23 15:22:35,752 - INFO - ====== WebDriver manager ======
2025-03-23 15:22:36,064 - INFO - Get LATEST geckodriver version for 136.0 firefox
2025-03-23 15:22:36,289 - INFO - Get LATEST geckodriver version for 136.0 firefox
2025-03-23 15:22:36,380 - INFO - Driver [/Users/magellan/.wdm/drivers/geckodriver/mac64/v0.36.0/geckodriver] found in cache


In [33]:
click_privacy_popup(driver)

2025-03-23 15:22:42,360 - INFO - Pop-up de confidentialité cliqué.


In [34]:
click_sort_button(driver)

2025-03-23 15:22:44,690 - INFO - Bouton de tri cliqué.


In [35]:
click_recent_reviews(driver)

2025-03-23 15:22:46,485 - INFO - Bouton 'Avis récents' cliqué.


In [36]:
get_number_of_reviews(driver)

2025-03-23 15:22:50,720 - INFO - Nombre total d'avis détectés : 160


160

In [39]:
scroll_reviews(driver)

2025-03-23 15:24:16,610 - INFO - Scrolling effectué.


In [18]:
extract_reviews(driver)

2025-03-23 15:19:01,413 - INFO - Nombre d'avis extraits : 10


[<div aria-label="Julien R." class="jftiEf fontBodyMedium" data-review-id="ChZDSUhNMG9nS0VJQ0FnTURRX3FXelBnEAE" jslog="21866;;metadata:WyIwYWhVS0V3anowNWpJc3FDTUF4VUFBR01CSGZxREVTZ1EwcE1GQ0FJb0FBIl0="><div data-review-id="ChZDSUhNMG9nS0VJQ0FnTURRX3FXelBnEAE" jsaction="mouseover:pane.wfvdle65.review.in; mouseout:pane.wfvdle65.review.out"><div class="jJc9Ad"><div> <button aria-label="Photo de Julien R." class="WEBjve" data-href="https://www.google.com/maps/contrib/107987219372922445992/reviews?hl=fr" data-review-id="ChZDSUhNMG9nS0VJQ0FnTURRX3FXelBnEAE" jsaction="pane.wfvdle65.review.reviewerLink" jslog="95010; track:click; mutable:true;"><img alt="" class="NBa7we" src="https://lh3.googleusercontent.com/a/ACg8ocLxxCT3GzYUjATViW90EFt9saxf36iJ1lrx2kQRb0SuKbAOgw=w72-h72-p-rp-mo-ba3-br100"/></button></div> <div class="GHT2ce NsCY4"><div class="m44Iib"></div> <div style="position: relative;"><div class="WNxzHc qLhwHc"><button class="al6Kxe" data-href="https://www.google.com/maps/contrib/107987

In [21]:
driver.close

<bound method WebDriver.close of <selenium.webdriver.firefox.webdriver.WebDriver (session="649f9e00-c793-437c-a648-a373ae12ed76")>>